<a href="https://colab.research.google.com/github/Nadsyuhamus/Databladez_Tourism/blob/feature%2Fml-tourism-engine/01_data_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01 — Tourism Data Audit

## Purpose

Understand the available tourism datasets before selecting features or training a model.

## Questions

1. What does each dataset represent?
2. What are the geographic and temporal coverage?
3. Which columns contain missing values?
4. Which variables can legitimately be used for forecasting?
5. Which data is state-level and which is district-level?
6. Is there data leakage or duplicated information?
7. What prediction target can be defended?

In [1]:
from pathlib import Path

REPO_URL = "https://github.com/Nadsyuhamus/Databladez_Tourism.git"
BRANCH = "feature/ml-tourism-engine"
REPO_DIR = Path("/content/Databladez_Tourism")

if not REPO_DIR.exists():
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull origin {BRANCH}

Cloning into '/content/Databladez_Tourism'...
remote: Enumerating objects: 124, done.
remote: Counting objects: 100% (124/124), done.
remote: Compressing objects: 100% (99/99), done.
remote: Total 124 (delta 47), reused 71 (delta 21), pack-reused 0 (from 0)
Receiving objects: 100% (124/124), 317.97 KiB | 4.75 MiB/s, done.
Resolving deltas: 100% (47/47), done.


In [5]:
DATA_DIR = (
    REPO_DIR
    / "outputs"
    / "datathon_cleaned_all_files"
)

print("Correct data directory:", DATA_DIR)
print("Directory exists:", DATA_DIR.exists())

Correct data directory: /content/Databladez_Tourism/outputs/datathon_cleaned_all_files
Directory exists: True


In [6]:
required_files = [
    "cleaned_data.csv",
    "state_tourism.csv",
    "district_context.csv",
    "google_trends_monthly.csv",
    "destination_rankings_2024_2025.csv",
    "data_dictionary.csv",
]

missing_files = []

for filename in required_files:
    file_path = DATA_DIR / filename

    if file_path.exists():
        size_mb = file_path.stat().st_size / (1024 ** 2)
        print(f"✓ {filename:<40} {size_mb:.2f} MB")
    else:
        missing_files.append(filename)
        print(f"✗ {filename}")

if missing_files:
    raise FileNotFoundError(
        f"Missing required files: {missing_files}"
    )

print("\nAll required datasets are available.")

✓ cleaned_data.csv                         0.03 MB
✓ state_tourism.csv                        0.02 MB
✓ district_context.csv                     0.08 MB
✓ google_trends_monthly.csv                1.15 MB
✓ destination_rankings_2024_2025.csv       0.02 MB
✓ data_dictionary.csv                      0.00 MB

All required datasets are available.


In [8]:
import pandas as pd
import numpy as np
from IPython.display import display

file_paths = {
    "cleaned_data": DATA_DIR / "cleaned_data.csv",
    "state_tourism": DATA_DIR / "state_tourism.csv",
    "district_context": DATA_DIR / "district_context.csv",
    "google_trends": DATA_DIR / "google_trends_monthly.csv",
    "destination_rankings": DATA_DIR / "destination_rankings_2024_2025.csv",
    "data_dictionary": DATA_DIR / "data_dictionary.csv",
}

datasets = {
    name: pd.read_csv(path)
    for name, path in file_paths.items()
}

print("Datasets loaded successfully:\n")

for name, dataframe in datasets.items():
    print(f"{name:<25} {dataframe.shape[0]:>6} rows × "
          f"{dataframe.shape[1]:>2} columns")

Datasets loaded successfully:

cleaned_data                 112 rows × 27 columns
state_tourism                112 rows × 15 columns
district_context             960 rows × 19 columns
google_trends              14810 rows ×  8 columns
destination_rankings         320 rows ×  5 columns
data_dictionary               27 rows ×  5 columns


In [9]:
audit_summary = []

for name, dataframe in datasets.items():
    total_cells = dataframe.shape[0] * dataframe.shape[1]
    missing_cells = int(dataframe.isna().sum().sum())

    audit_summary.append({
        "dataset": name,
        "rows": dataframe.shape[0],
        "columns": dataframe.shape[1],
        "duplicate_rows": int(dataframe.duplicated().sum()),
        "missing_cells": missing_cells,
        "missing_percentage": (
            round(missing_cells / total_cells * 100, 2)
            if total_cells > 0
            else 0
        ),
    })

audit_summary = pd.DataFrame(audit_summary)

display(audit_summary)

,dataset,rows,columns,duplicate_rows,missing_cells,missing_percentage
0,cleaned_data,112,27,0,425,14.05
1,state_tourism,112,15,0,0,0.00
2,district_context,960,19,0,5807,31.84
3,google_trends,14810,8,0,14130,11.93
4,destination_rankings,320,5,0,0,0.00
5,data_dictionary,27,5,0,17,12.59


In [10]:
coverage_summary = pd.DataFrame([
    {
        "dataset": "cleaned_data",
        "geographic_level": "State",
        "locations": datasets["cleaned_data"]["state"].nunique(),
        "start_year": datasets["cleaned_data"]["year"].min(),
        "end_year": datasets["cleaned_data"]["year"].max(),
    },
    {
        "dataset": "state_tourism",
        "geographic_level": "State",
        "locations": datasets["state_tourism"]["state"].nunique(),
        "start_year": datasets["state_tourism"]["year"].min(),
        "end_year": datasets["state_tourism"]["year"].max(),
    },
    {
        "dataset": "district_context",
        "geographic_level": "District",
        "locations": datasets["district_context"]["district"].nunique(),
        "start_year": datasets["district_context"]["year"].min(),
        "end_year": datasets["district_context"]["year"].max(),
    },
    {
        "dataset": "google_trends",
        "geographic_level": "Mixed state/area",
        "locations": datasets["google_trends"]["area"].nunique(),
        "start_year": datasets["google_trends"]["year"].min(),
        "end_year": datasets["google_trends"]["year"].max(),
    },
    {
        "dataset": "destination_rankings",
        "geographic_level": "Destination",
        "locations": datasets["destination_rankings"]["destination"].nunique(),
        "start_year": datasets["destination_rankings"]["year"].min(),
        "end_year": datasets["destination_rankings"]["year"].max(),
    },
])

display(coverage_summary)

,dataset,geographic_level,locations,start_year,end_year
0,cleaned_data,State,16,2019,2025
1,state_tourism,State,16,2019,2025
2,district_context,District,163,2020,2025
3,google_trends,Mixed state/area,167,2018,2025
4,destination_rankings,Destination,164,2024,2025


In [11]:
for name, dataframe in datasets.items():
    missing = (
        dataframe.isna()
        .sum()
        .to_frame("missing_count")
    )

    missing["missing_percentage"] = (
        missing["missing_count"] / len(dataframe) * 100
    ).round(2)

    missing = missing[
        missing["missing_count"] > 0
    ].sort_values("missing_percentage", ascending=False)

    print(f"\n{name.upper()} — missing values")

    if missing.empty:
        print("No missing values.")
    else:
        display(missing)


CLEANED_DATA — missing values


,missing_count,missing_percentage
income_mean,84,75.00
income_median,84,75.00
expenditure_mean,84,75.00
gini,84,75.00
poverty,84,75.00
search_interest_growth_pct,5,4.46



STATE_TOURISM — missing values
No missing values.

DISTRICT_CONTEXT — missing values


,missing_count,missing_percentage
gdp_real_p0,801,83.44
income_median,652,67.92
income_mean,652,67.92
gini,652,67.92
poverty,652,67.92
expenditure_mean,652,67.92
google_trend_yoy_change,229,23.85
ep_ratio,223,23.23
lf,223,23.23
lf_employed,198,20.62



GOOGLE_TRENDS — missing values


,missing_count,missing_percentage
duplicate_column_flag,14130,95.41



DESTINATION_RANKINGS — missing values
No missing values.

DATA_DICTIONARY — missing values


,missing_count,missing_percentage
unit,12,44.44
description,5,18.52


# Forecasting Objective

## Primary task

Predict the next month's Google Trends tourism search-interest index for each Malaysian state or area series.

## Target

`target_next_month`

This represents the next calendar month's `trend_index`.

## Prediction horizon

- Primary: one month ahead
- Possible extension: three months ahead

## Important interpretation

The model forecasts digital tourism-interest momentum, not actual tourist arrivals.

Google Trends values are relative indices. A value from one independently
scaled geographic series should not automatically be interpreted as directly
comparable with another series.

## Validation requirement

Training and testing must follow chronological order. Random train-test
splitting is prohibited because it would leak future information.

In [12]:
trends = datasets["google_trends"].copy()

trends["month"] = pd.to_datetime(
    trends["month"],
    errors="coerce"
)

trends["trend_index"] = pd.to_numeric(
    trends["trend_index"],
    errors="coerce"
)

trends["series_id"] = (
    trends[["state", "geo_level", "area"]]
    .fillna("unknown")
    .astype(str)
    .agg(" | ".join, axis=1)
)

trends = trends.sort_values(
    ["series_id", "month"]
).reset_index(drop=True)

print("Rows:", len(trends))
print("Unique series:", trends["series_id"].nunique())
print("Start month:", trends["month"].min())
print("End month:", trends["month"].max())
print("Missing dates:", trends["month"].isna().sum())
print("Missing trend values:", trends["trend_index"].isna().sum())

Rows: 14810
Unique series: 167
Start month: 2018-12-01 00:00:00
End month: 2025-12-01 00:00:00
Missing dates: 0
Missing trend values: 0


In [13]:
geo_summary = (
    trends.groupby("geo_level")
    .agg(
        rows=("trend_index", "size"),
        unique_series=("series_id", "nunique"),
        states=("state", "nunique"),
        start_month=("month", "min"),
        end_month=("month", "max"),
        missing_trend_values=("trend_index", lambda x: x.isna().sum()),
    )
    .reset_index()
)

display(geo_summary)

,geo_level,rows,unique_series,states,start_month,end_month,missing_trend_values
0,district_or_area,13370,151,12,2018-12-01,2025-12-01,0
1,state,1440,16,16,2018-12-01,2025-12-01,0


In [15]:
duplicate_keys = trends.duplicated(
    subset=["series_id", "month"],
    keep=False
)

print(
    "Duplicate series-month rows:",
    duplicate_keys.sum()
)

if duplicate_keys.any():
    display(
        trends.loc[
            duplicate_keys,
            [
                "state",
                "area",
                "geo_level",
                "month",
                "trend_index",
                "source_file",
                "series_id", # Added series_id to the list of displayed columns
            ],
        ].sort_values(["series_id", "month"])
    )

Duplicate series-month rows: 1360


,state,area,geo_level,month,trend_index,source_file,series_id
7054,Sabah,Kinabatangan,district_or_area,2018-12-01,3.0,time_series_MY_SABAH.csv,Sabah | district_or_area | Kinabatangan
7055,Sabah,Kinabatangan,district_or_area,2018-12-01,3.0,time_series_MY_SABAH.csv,Sabah | district_or_area | Kinabatangan
7056,Sabah,Kinabatangan,district_or_area,2019-01-01,3.0,time_series_MY_SABAH.csv,Sabah | district_or_area | Kinabatangan
7057,Sabah,Kinabatangan,district_or_area,2019-01-01,3.0,time_series_MY_SABAH.csv,Sabah | district_or_area | Kinabatangan
7058,Sabah,Kinabatangan,district_or_area,2019-02-01,4.0,time_series_MY_SABAH.csv,Sabah | district_or_area | Kinabatangan
...,...,...,...,...,...,...,...
9514,Sabah,Sabah,state,2025-10-01,86.0,time_series_MY_SABAH.csv,Sabah | state | Sabah
9515,Sabah,Sabah,state,2025-11-01,73.0,time_series_MY_SABAH.csv,Sabah | state | Sabah
9516,Sabah,Sabah,state,2025-11-01,73.0,time_series_MY_SABAH.csv,Sabah | state | Sabah
9517,Sabah,Sabah,state,2025-12-01,41.0,time_series_MY_SABAH.csv,Sabah | state | Sabah


In [16]:
series_records = []

for series_id, group in trends.groupby("series_id"):
    valid_months = (
        group["month"]
        .dropna()
        .drop_duplicates()
        .sort_values()
    )

    if valid_months.empty:
        expected_months = 0
    else:
        expected_months = len(
            pd.date_range(
                start=valid_months.min(),
                end=valid_months.max(),
                freq="MS",
            )
        )

    observed_months = len(valid_months)

    series_records.append({
        "series_id": series_id,
        "state": group["state"].iloc[0],
        "area": group["area"].iloc[0],
        "geo_level": group["geo_level"].iloc[0],
        "start_month": valid_months.min()
            if not valid_months.empty else pd.NaT,
        "end_month": valid_months.max()
            if not valid_months.empty else pd.NaT,
        "observed_months": observed_months,
        "expected_months": expected_months,
        "missing_months": expected_months - observed_months,
        "coverage_percentage": round(
            observed_months / expected_months * 100,
            2,
        ) if expected_months else 0,
        "unique_trend_values": group["trend_index"].nunique(),
    })

series_audit = pd.DataFrame(series_records)

display(series_audit.head())

,series_id,state,area,geo_level,start_month,end_month,observed_months,expected_months,missing_months,coverage_percentage,unique_trend_values
0,Johor | district_or_area | Batu Pahat,Johor,Batu Pahat,district_or_area,2018-12-01,2025-12-01,85,85,0,100.0,14
1,Johor | district_or_area | Kluang,Johor,Kluang,district_or_area,2018-12-01,2025-12-01,85,85,0,100.0,6
2,Johor | district_or_area | Kota Tinggi,Johor,Kota Tinggi,district_or_area,2018-12-01,2025-12-01,85,85,0,100.0,39
3,Johor | district_or_area | Kulai,Johor,Kulai,district_or_area,2018-12-01,2025-12-01,85,85,0,100.0,9
4,Johor | district_or_area | Mersing,Johor,Mersing,district_or_area,2018-12-01,2025-12-01,85,85,0,100.0,42


In [17]:
print("Total series:", len(series_audit))

print(
    "Series with at least 36 observations:",
    (series_audit["observed_months"] >= 36).sum()
)

print(
    "Series with at least 90% monthly coverage:",
    (series_audit["coverage_percentage"] >= 90).sum()
)

print(
    "Constant or near-constant series:",
    (series_audit["unique_trend_values"] <= 1).sum()
)

display(
    series_audit[
        [
            "observed_months",
            "missing_months",
            "coverage_percentage",
            "unique_trend_values",
        ]
    ].describe().round(2)
)

Total series: 167
Series with at least 36 observations: 167
Series with at least 90% monthly coverage: 167
Constant or near-constant series: 35


,observed_months,missing_months,coverage_percentage,unique_trend_values
count,167.00,167.00,167.00,167.00
mean,84.61,0.12,99.86,11.00
std,1.03,0.83,0.99,14.38
min,76.00,0.00,90.48,1.00
25%,84.00,0.00,100.00,2.00
50%,85.00,0.00,100.00,4.00
75%,85.00,0.00,100.00,13.50
max,85.00,8.00,100.00,54.00


In [18]:
trends["next_observed_month"] = (
    trends.groupby("series_id")["month"]
    .shift(-1)
)

shifted_target = (
    trends.groupby("series_id")["trend_index"]
    .shift(-1)
)

is_next_calendar_month = (
    trends["next_observed_month"]
    == trends["month"] + pd.offsets.MonthBegin(1)
)

trends["target_next_month"] = shifted_target.where(
    is_next_calendar_month
)

print(
    "Rows with a valid next-month target:",
    trends["target_next_month"].notna().sum()
)

print(
    "Rows without a target:",
    trends["target_next_month"].isna().sum()
)

display(
    trends[
        [
            "series_id",
            "month",
            "trend_index",
            "next_observed_month",
            "target_next_month",
        ]
    ].head(15)
)

Rows with a valid next-month target: 13945
Rows without a target: 865


,series_id,month,trend_index,next_observed_month,target_next_month
0,Johor | district_or_area | Batu Pahat,2018-12-01,16.0,2019-01-01,11.0
1,Johor | district_or_area | Batu Pahat,2019-01-01,11.0,2019-02-01,17.0
2,Johor | district_or_area | Batu Pahat,2019-02-01,17.0,2019-03-01,14.0
3,Johor | district_or_area | Batu Pahat,2019-03-01,14.0,2019-04-01,12.0
4,Johor | district_or_area | Batu Pahat,2019-04-01,12.0,2019-05-01,13.0
5,Johor | district_or_area | Batu Pahat,2019-05-01,13.0,2019-06-01,14.0
6,Johor | district_or_area | Batu Pahat,2019-06-01,14.0,2019-07-01,12.0
7,Johor | district_or_area | Batu Pahat,2019-07-01,12.0,2019-08-01,15.0
8,Johor | district_or_area | Batu Pahat,2019-08-01,15.0,2019-09-01,14.0
9,Johor | district_or_area | Batu Pahat,2019-09-01,14.0,2019-10-01,13.0
